# Human-in-the-Loop Interrupt Gates

## Problem Statement
Wire a HITL interrupt gate into the DevContext AI PR impact branch - pause on low-confidence caller matches and resume based on engineer input.

## My Approach

I drew the updated graph on paper first. langgraph-agent.ipynb had `diff_parser -> dependency_finder -> impact_generate`. This code adds a pause point between finding callers and generating the report, so I inserted a new `find_callers` node between `dependency_finder` and `impact_generate`.

For state, I added three new fields: `high_confidence_callers` and `uncertain_candidates` to hold the split results, and `user_approval` to carry the engineer's `y/n` decision across the interrupt boundary. I kept them in state (not local variables) because I knew `interrupt()` freezes execution - anything not in state would be gone on resume.

The interrupt logic I put in `find_callers`: check if `user_approval is None` and `uncertain_candidates` is non-empty -> call `interrupt()`. After `Command(resume=...)`, the node re-enters with `user_approval` set, hits the guard condition, and passes through. `impact_generate` then reads `user_approval` and builds the final list from already-computed state - no re-querying.

For the runner, I used `app.stream()` with a `for` loop watching for `__interrupt__` in the event dict.

## What I Learned

- `interrupt()` is not a special node type - it's a function call inside a regular node. The node re-executes on resume; the return value of `interrupt()` is whatever was passed to `Command(resume=...)`. So the same node handles both the pause and the post-resume logic.
- `MemorySaver` is a hard prerequisite - not optional config. Without a checkpointer, `interrupt()` has nowhere to serialize state and crashes.
- `thread_id` is the checkpoint key. Resuming with a different `thread_id` starts a fresh run silently - no error, no warning. This is the easiest bug to miss in production.
- `app.invoke()` is cleaner than `app.stream()` for interrupt detection. `stream()` emits intermediate snapshots; `invoke()` returns once (interrupted state or final state) and is unambiguous.
- State is the only memory that survives an interrupt. Local variables inside a node do not.

## Where I Got Stuck

**The query mismatch.** I passed `"Renamed process_payment() -> execute_payment()"` as the query, which has no `+/-def` lines, so `diff_parser` extracted `changed_functions = []`, `dependency_finder` returned `"No changes detected"`, and the regex path was completely bypassed. My `find_callers` then ran on `MOCK_CANDIDATES` anyway, so results appeared - but `retrieved_context` was wrong. I should have used `SAMPLE_DIFF` (same as langgraph-agent.ipynb) from the start.

## What I'd Do Differently

- **Use `app.invoke()` for interrupt detection** instead of a `for event in app.stream()` loop with a `break`. The stream pattern works but is fragile - `event` at the break point is a mid-stream snapshot, and the `else` clause on a `for` loop is not intuitive for error-path reasoning.
- **Add `empty_state()` helper immediately.** Initialising every TypedDict key in one place prevents the `""` vs `[]` class of bug I hit on Day 18 and avoids `KeyError` on any node that does `.copy()` or direct key access on HITL fields.
- **Test the no-interrupt path explicitly.** I only tested the low-confidence trigger. If all candidates were above threshold, `find_callers` returned without setting `high_confidence_callers` - `impact_generate` would have crashed with `KeyError`. A one-line test with all high-confidence mock data would have caught this immediately.

In [ ]:
MOCK_CODEBASE = {
    "authenticate_user": """
def authenticate_user(user_id: str) -> dict:
    \"\"\"
    Validates user credentials against the users table.
    Fetches user record, checks account_status != 'suspended',
    generates a signed JWT with 24h expiry, logs auth event to audit_log.
    Raises AuthError if user not found or suspended.
    Returns: {"token": str, "expires_at": datetime, "user_id": str}
    \"\"\"
""",
    "get_user_profile": """
def get_user_profile(user_id: str, include_prefs: bool = False) -> dict:
    \"\"\"
    Fetches user metadata from users + user_preferences tables.
    Calls authenticate_user(user_id) internally to validate session.
    \"\"\"
""",
    "create_order": """
def create_order(user_id: str, items: list[dict], payment_method_id: str) -> dict:
    \"\"\"
    Validates cart, reserves inventory, calls authenticate_user(user_id) before processing.
    \"\"\"
""",
    "refresh_token": """
def refresh_token(expired_token: str) -> dict:
    \"\"\"
    Calls authenticate_user(user_id) to revalidate account status.
    Issues a new token. Blacklists the old one in Redis.
    \"\"\"
"""
}

MOCK_DOCS = {
    "auth-flow.md": """
# Authentication Flow
- Issued by `authenticate_user()` in `auth.py`
- Expiry: 24h. Refresh via `POST /auth/refresh`
- 401 Unauthorized - token missing or malformed
- 403 Forbidden - valid token but suspended account
""",
}

SAMPLE_DIFF = """
--- a/auth.py
+++ b/auth.py
@@ -12,7 +12,7 @@
-def authenticate_user(user_id: str) -> dict:
+def authenticate_user(user_id: str, token: str = None) -> dict:
     \"\"\"
-    Validates user credentials. Generates JWT on success.
+    Validates user credentials. If token provided, validates existing session.
"""

# Simulated ChromaDB scores - keyed to MOCK_CODEBASE function names + one external file
# In production: chromadb .query() returns distances alongside documents
MOCK_CANDIDATES = {
    "get_user_profile":     {"file": "auth/profile.py",       "function": "get_user_profile", "score": 0.91},
    "create_order":         {"file": "payments/checkout.py",  "function": "run_checkout",     "score": 0.88},
    "refresh_token":        {"file": "billing/refunds.py",    "function": "issue_refund",     "score": 0.85},
    # Borderline - possible indirect caller via middleware; below threshold
    "admin_billing_report": {"file": "admin/reports.py",      "function": "billing_summary",  "score": 0.61},
}

CONFIDENCE_THRESHOLD = 0.75

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START , END
from langchain_groq import ChatGroq
import re
from typing import TypedDict, Annotated, List, Optional
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

class AgentState(TypedDict):
    query: str              # "user query"
    mode: str               # "qa" or "impact"
    retrieved_context: str  # "docs/code chunks retrieved from ChromaDB"
    answer: str             # llm answer
    sources: list[str]      # "Sources retrieved from chromadb"
    changed_functions: list[str] # extracted from PR diff
    impacted_files: list[str]   # files that call changed functions
    # --- New HITL Fields ---
    high_confidence_callers: list[dict]
    uncertain_candidates: list[dict]
    user_approval: Optional[str]

#Refactored solution
def empty_state(query: str) -> AgentState:
    """Always initialise ALL keys - prevents KeyError in any node."""
    return {
        "query": query, "mode": "", "retrieved_context": "",
        "answer": "", "sources": [], "changed_functions": [],
        "impacted_files": [], "high_confidence_callers": [],
        "uncertain_candidates": [], "user_approval": None,
    }

In [ ]:
#entry node
def route_intent(state: AgentState) -> dict :
    prompt = f"""
    Classify below query into qa or pr_impact .
    qa=asking about how code/system works or onboarding questions
    pr_impact=asking about what code breaks if code changes are done , pr review questions.
    Query = {state['query']}
    Respond with one word only : qa or pr_impact.
    """
    response = llm.invoke(prompt , max_tokens=5)
    mode=response.content.strip().lower()
    if mode not in ('qa','pr_impact'):
        mode='qa' #fallback
    return {"mode":mode}

#Mapping intent 
def intent_router(state: AgentState) -> str:
    return state["mode"]  # "qa" -> qa branch, "pr_impact" ->  pr branch 

#QA answer generate , retrieval would be from chromadb and it would have been separate function in production case.
def qa_retrieve_generate(state: AgentState) -> dict:
    query_lower = state["query"].lower()
    relevant_code = {
        k: v for k, v in MOCK_CODEBASE.items()
        if any(word in query_lower for word in k.split("_")) or k in query_lower
    }
    if not relevant_code:
        first_key = next(iter(MOCK_CODEBASE))
        relevant_code = {first_key: MOCK_CODEBASE[first_key]}

    relevant_docs = {
        k: v for k, v in MOCK_DOCS.items()
        if any(word in v.lower() for word in query_lower.split())
    }

    context_code_str = "\n".join(f"[codebase:{k}]\n{v}" for k, v in relevant_code.items())
    context_doc_str  = "\n".join(f"[docs:{k}]\n{v}"     for k, v in relevant_docs.items())

    prompt = (
        "You are a codebase assistant. Answer using ONLY the context below.\n"
        "Cite source keys used (e.g. codebase:authenticate_user, docs:auth-flow.md).\n\n"
        f"--- Codebase ---\n{context_code_str}\n\n"
        f"--- Docs ---\n{context_doc_str}\n\n"
        f"Query: {state['query']}\n\n"
        "Output format (two lines only):\n"
        "Sources: comma-separated source keys\n"
        "Answer: your answer"
    )
    response = llm.invoke(prompt)
    content = response.content.strip()

    sources, answer = [], content
    for line in content.splitlines():
        if line.lower().startswith("sources:"):
            sources = [s.strip() for s in line.split(":", 1)[1].split(",")]
        elif line.lower().startswith("answer:"):
            answer = line.split(":", 1)[1].strip()

    return {
        "retrieved_context": context_code_str + "\n" + context_doc_str,
        "answer": answer,
        "sources": sources,
    }

#pr impact code.
def diff_parser (state : AgentState) -> dict:
    pattern = r'^[+-][\t ]*def\s+(\w+)\s*\('
    matches=re.findall(pattern , state['query'] , re.MULTILINE)
    return {"changed_functions" :list(set(matches))}

In [ ]:
def dependency_finder(state: AgentState) -> dict:
    changed_fns = state.get("changed_functions", [])
    if not changed_fns:
        return {
            "retrieved_context": "No changes detected",
            "impacted_files": [],
            "high_confidence_callers": [],
            "uncertain_candidates": [],
        }

    high_confidence = []
    uncertain = []
    context_chunks = []

    for fn_name in changed_fns:
        call_pattern = re.compile(fr'\b{re.escape(fn_name.strip())}\s*\(')
        for key, code_content in MOCK_CODEBASE.items():
            if call_pattern.search(code_content):
                candidate = MOCK_CANDIDATES.get(key)
                if candidate is None:
                    # No score available - treat as high confidence (safe default)
                    candidate = {"file": key, "function": key, "score": 0.99}
                score = candidate["score"]
                context_chunks.append(f"--- {candidate['file']} (score={score}) ---\n{code_content}")
                if score >= CONFIDENCE_THRESHOLD:
                    high_confidence.append(candidate)
                else:
                    uncertain.append(candidate)

    # To include any MOCK_CANDIDATES entries that are NOT in MOCK_CODEBASE
    # (e.g. admin/reports.py - indirect caller, no local code to regex-match against)
    regex_matched_keys = set()
    for fn_name in changed_fns:
        p = re.compile(fr'\b{re.escape(fn_name.strip())}\s*\(')
        for key, code_content in MOCK_CODEBASE.items():
            if p.search(code_content):
                regex_matched_keys.add(key)

    for key, candidate in MOCK_CANDIDATES.items():
        if key not in regex_matched_keys:
            score = candidate["score"]
            context_chunks.append(
                f"--- {candidate['file']} (score={score}, indirect/external) ---\n"
                f"[Not found in local codebase - possible indirect caller]"
            )
            if score >= CONFIDENCE_THRESHOLD:
                high_confidence.append(candidate)
            else:
                uncertain.append(candidate)

    return {
        "retrieved_context": "\n".join(context_chunks),
        "impacted_files": [c["file"] for c in high_confidence],
        "high_confidence_callers": high_confidence,
        "uncertain_candidates": uncertain,
    }


def find_callers(state: AgentState) -> dict:
    uncertain = state.get("uncertain_candidates", [])

    # Pass-through if nothing uncertain or already resolved
    if not uncertain or state.get("user_approval") is not None:
        return {}

    high_conf = state.get("high_confidence_callers", [])

    decision = interrupt({
        "message": "INTERRUPT - Low-confidence match(es) detected",
        "candidates": uncertain,
        "current_high_confidence": high_conf,
        "threshold": CONFIDENCE_THRESHOLD,
    })

    return {"user_approval": decision}


def impact_generate(state: AgentState) -> dict:
    final_list = list(state.get("high_confidence_callers", []))
    decision = state.get("user_approval", "n")

    if decision and decision.strip().lower() == "y":
        final_list.extend(state.get("uncertain_candidates", []))

    uncertain = state.get("uncertain_candidates", [])
    report_lines = ["\n=== Final PR Impact Report ==="]
    for c in final_list:
        report_lines.append(f"{c['file']}  -  {c['function']}  (score: {c['score']})")
    if decision and decision.strip().lower() == "n":
        for c in uncertain:
            report_lines.append(f"{c['file']}  -  dropped by engineer  (score: {c['score']})")
    report_lines.append("==============================")

    prompt = (
        "You are a code impact analyst. Explain what could break and why, "
        "grounded in the context and confirmed files only. Two sentences per file max.\n\n"
        f"Context:\n{state['retrieved_context']}\n\n"
        f"Confirmed impacted files: {[c['file'] for c in final_list]}"
    )
    response = llm.invoke(prompt)
    return {
        "impacted_files": [c["file"] for c in final_list],
        "answer": "\n".join(report_lines) + "\n\n" + response.content,
    }

In [21]:
memory = MemorySaver()
graph = StateGraph(AgentState)
graph.add_node('route_intent' , route_intent)
# graph.add_node("intent_router", intent_router)
graph.add_node('qa_retrieve_generate' , qa_retrieve_generate)
graph.add_node('diff_parser' , diff_parser)
graph.add_node('dependency_finder' , dependency_finder)
graph.add_node("find_callers", find_callers) # New HITL Node
graph.add_node('impact_generate' , impact_generate)

graph.add_edge(START, 'route_intent')
graph.add_conditional_edges('route_intent' , intent_router , 
{
        "qa": "qa_retrieve_generate", 
        "pr_impact": "diff_parser"
})

graph.add_edge('qa_retrieve_generate' , END)
graph.add_edge('diff_parser' , 'dependency_finder')
graph.add_edge('dependency_finder' , 'find_callers')
graph.add_edge('find_callers' , 'impact_generate')
graph.add_edge('impact_generate' , END)

app=graph.compile(checkpointer=memory)


In [22]:
# Visualize
print(app.get_graph().draw_ascii())

                    +-----------+                        
                    | __start__ |                        
                    +-----------+                        
                           *                             
                           *                             
                           *                             
                   +--------------+                      
                   | route_intent |                      
                   +--------------+                      
                  ..              ...                    
               ...                   ..                  
             ..                        ...               
   +-------------+                        ..             
   | diff_parser |                         .             
   +-------------+                         .             
          *                                .             
          *                                .             
          *   

In [ ]:
thread_yes = {"configurable": {"thread_id": "pr-review-yes-01"}}

print("[Agent running PR impact analysis...]")

#Refactored solution
state_1 = app.invoke(empty_state(SAMPLE_DIFF), config=thread_yes)

if "__interrupt__" in state_1:
    payload = state_1["__interrupt__"][0].value
    print(f"\n{payload['message']}")
    for c in payload["candidates"]:
        print(f"  File: {c['file']}  |  Function: {c['function']}  |  Score: {c['score']}")
    print(f"  Threshold: {payload['threshold']}")
    print("\n  High-confidence callers already confirmed:")
    for c in payload["current_high_confidence"]:
        print(f"{c['file']}  ({c['score']})")

    choice = input("\nInclude this file in the impact report? (y/n): ").strip().lower()
    print("\n[Resuming graph with human decision: YES]")
    final = app.invoke(Command(resume=choice), config=thread_yes)
    print(final["answer"])
else:
    print("No interrupt fired.")
    print(state_1["answer"])

[Agent running PR impact analysis...]

INTERRUPT — Low-confidence match(es) detected
  File: admin/reports.py  |  Function: billing_summary  |  Score: 0.61
  Threshold: 0.75

  High-confidence callers already confirmed:
authenticate_user  (0.99)
auth/profile.py  (0.91)
payments/checkout.py  (0.88)
billing/refunds.py  (0.85)

[Resuming graph with human decision: YES]

=== Final PR Impact Report ===
authenticate_user  -  authenticate_user  (score: 0.99)
auth/profile.py  -  get_user_profile  (score: 0.91)
payments/checkout.py  -  run_checkout  (score: 0.88)
billing/refunds.py  -  issue_refund  (score: 0.85)
admin/reports.py  -  billing_summary  (score: 0.61)

Based on the provided context and confirmed files, potential issues could arise as follows:

**authenticate_user**: The `authenticate_user` function could break if the database connection to the users table fails, causing it to be unable to fetch user records or log auth events to the audit log. Additionally, if the JWT signing proce

In [26]:
thread_no = {"configurable": {"thread_id": "pr-review-no-01"}}

print("[Agent running PR impact analysis...]")
state_2 = app.invoke(empty_state(SAMPLE_DIFF), config=thread_no)

if "__interrupt__" in state_2:
    payload = state_2["__interrupt__"][0].value
    print(f"\n{payload['message']}")
    for c in payload["candidates"]:
        print(f"  File: {c['file']}  |  Score: {c['score']}")

    choice = input("\nInclude this file in the impact report? (y/n): ").strip().lower()
    print("\n[Resuming graph with human decision: NO]")
    final = app.invoke(Command(resume=choice), config=thread_no)
    print(final["answer"])
else:
    print(state_2["answer"])

[Agent running PR impact analysis...]

INTERRUPT — Low-confidence match(es) detected
  File: admin/reports.py  |  Score: 0.61

[Resuming graph with human decision: NO]

=== Final PR Impact Report ===
authenticate_user  -  authenticate_user  (score: 0.99)
auth/profile.py  -  get_user_profile  (score: 0.91)
payments/checkout.py  -  run_checkout  (score: 0.88)
billing/refunds.py  -  issue_refund  (score: 0.85)
admin/reports.py  -  dropped by engineer  (score: 0.61)

Based on the provided context and confirmed files, potential issues could arise as follows:

In `authenticate_user`, the function could break if the users table is not properly updated or if the JWT generation process fails, causing authentication errors. The function's reliance on the `users` table and JWT generation also makes it vulnerable to database connection issues or library updates that affect JWT creation.

In `auth/profile.py`, the `get_user_profile` function could break if the `authenticate_user` function fails or 